# Problem Set 7: Social Media and Mental Health

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 11/11 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

A staggered difference-in-differences and event-study analysis of Facebook's rollout across US colleges.

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

This problem set is based on Braghieri, Levy, and Makarin (2022), "Social Media and Mental Health," *American Economic Review* 112(11): 3660–3693. The question: **did the introduction of Facebook harm college students' mental health?**

That is hard to answer because social media use is not randomly assigned — people who are struggling might use (or avoid) it differently, biasing any simple correlation. The authors' solution is a natural experiment: Facebook rolled out across US colleges in a **staggered** fashion between February 2004 and September 2006 (Harvard first, then a widening set), well before it opened to the public. Combining that rollout with mental health surveys from the National College Health Assessment, they estimate a **generalized difference-in-differences** model: for each college, compare mental health before versus after Facebook arrived, and compare that within-college change to the same change at colleges where Facebook arrived later.

Their headline finding: Facebook increased an index of poor mental health by **0.085 standard deviations**.

> **A note on the data.** The NCHA microdata behind the paper are restricted-use, so we cannot hand you the real dataset. This is a **simulated dataset** that (a) uses the *real* Facebook introduction dates for 150 US colleges from the paper's replication files, and (b) simulates plausible student-level responses calibrated so you should recover something close to 0.085 SD. Your estimate will differ somewhat from the paper's. The point is to practice the methods, not to reproduce exact numbers.

| Variable | Description |
|---|---|
| `uni_id`, `college_name`, `region` | College identifier, name, census region |
| `fbexpgrp` | Facebook expansion batch (1 = earliest … 4 = latest) |
| `fb_intro_date`, `survey_wave`, `wave_date`, `year`, `fall_survey` | Rollout date and survey timing |
| `post` / `T` | =1 if the wave occurred after Facebook arrived at that college |
| `exposure_months` | Months since Facebook arrived (0 if not yet) |
| `age`, `age_sq`, `female`, `year_in_school`, `white`, `black`, `hispanic`, `asian`, `indian`, `other_race`, `international` | Student controls |
| `eq_index_mh_all` | **Outcome:** standardized index of poor mental health (higher = worse) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

# With 150 college dummies and clustered standard errors, statsmodels prints a
# harmless warning about the rank of the overall F-test. Our coefficient and
# standard errors are unaffected, so we silence it to keep the output readable.
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings("ignore", category=ValueWarning)

print("Ready!")

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/ps_b_socialmedia_mh.csv")
print(df.shape)
df.head()

## Part 1: Understanding the design

**Question 1.** Describe the data: how many colleges, how many survey waves, and roughly how many student observations per college-wave cell?

In [ ]:
print("Colleges:      ", df.uni_id.nunique())
print("Survey waves:  ", df.survey_wave.nunique())
print("Observations:  ", f"{len(df):,}")
print("Per college-wave cell:", round(len(df) / (df.uni_id.nunique() * df.survey_wave.nunique()), 1))

**Answer:** *[your answer here]*

**Question 2. Explain the identifying variation in your own words.** Why is it not enough to compare mental health at colleges that had Facebook to colleges that did not, at a single point in time? Why is it not enough to compare a single college before and after Facebook arrived, with no comparison group? **What does the DiD design add on top of each of these simpler comparisons?**

**Answer:** *[your answer here]*

**Question 3.** Tabulate colleges by expansion group, then show mean mental health by `fbexpgrp` and `post` in a 4×2 table. **Just from this simple table, does it look like mental health worsened after Facebook arrived?**

In [ ]:
print("Colleges per expansion group:")
print(df.groupby("fbexpgrp").uni_id.nunique(), "\n")

df.pivot_table(index="fbexpgrp", columns="post",
               values="eq_index_mh_all", aggfunc="mean").round(3)

**Answer:** *[your answer here]*

## Part 2: The difference-in-differences regression

**Question 4.** Run the baseline two-way fixed effects DiD: outcome `eq_index_mh_all`, treatment `T`, with **college fixed effects** and **survey-wave fixed effects**, clustering standard errors at the college level (treatment varies at the college level, and students at the same college in the same period are likely correlated).

*This regression estimates 150 college dummies on 102,000 observations, so it takes a minute to run.*

In [ ]:
did1 = smf.ols("eq_index_mh_all ~ T + C(uni_id) + C(survey_wave)", df).fit(
    cov_type="cluster", cov_kwds={"groups": df.uni_id})

print(did1.summary().tables[1].as_text().split("\n")[0])
print(did1.summary().tables[1].as_text().split("\n")[2])   # the T row
print(f"\nEffect of Facebook: {did1.params['T']:.4f} SD "
      f"(se {did1.bse['T']:.4f}, t = {did1.tvalues['T']:.2f})")
print(f"Paper's estimate:   0.0850 SD")

**Question 4 (continued).** Report and interpret the coefficient on `T`. Is it statistically significant? How does its magnitude compare to the paper's 0.085 SD?

**Answer:** *[your answer here]*

**Question 5.** Add the demographic controls and re-run, then present both regressions side by side (the equivalent of `esttab` with two columns).

In [ ]:
controls = ("age + age_sq + female + year_in_school + black + hispanic + asian "
            "+ indian + other_race + international")

did2 = smf.ols(f"eq_index_mh_all ~ T + {controls} + C(uni_id) + C(survey_wave)", df).fit(
    cov_type="cluster", cov_kwds={"groups": df.uni_id})

print(summary_col([did1, did2], stars=True, float_format="%.4f",
                  model_names=["(1) FE only", "(2) FE + controls"],
                  regressor_order=["T"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}",
                             "College FE": lambda r: "Yes",
                             "Wave FE": lambda r: "Yes"}))

**Question 6. Why do we include college fixed effects?** What confound would we fail to address without them? Give a concrete example of a college-level characteristic correlated with both Facebook's rollout timing and student mental health.

**Answer:** *[your answer here]*

**Question 7. Why do we include survey-wave fixed effects?** What confound would we fail to address without them? Give a concrete example.

**Answer:** *[your answer here]*

**Question 8. What identifying assumption does this design still rely on, even with both sets of fixed effects?** (Think about what would happen if colleges that got Facebook earlier were already on a different mental-health *trajectory* than later colleges, for unrelated reasons.) This is the **parallel trends** assumption.

**Answer:** *[your answer here]*

## Part 3: The event-study graph

The regression above gives only the *average* post-Facebook effect, pooling colleges shortly after arrival with colleges years later. It also cannot tell you whether pre-existing trends were similar — exactly the parallel-trends assumption. An **event study** addresses both by estimating a separate coefficient for each period *relative to treatment*.

**Question 9.** For each college find the first treated wave, build `event_time = survey_wave - intro_wave`, bin the tails at ±6, and estimate the event-study regression omitting `event_time = -1` as the reference period.

In [ ]:
# first wave in which each college is treated
intro = (df[df.post == 1].groupby("uni_id").survey_wave.min().rename("intro_wave"))
df = df.merge(intro, on="uni_id", how="left")
df["event_time"] = df.survey_wave - df.intro_wave

# bin the tails: few colleges identify the extreme event times
df["event_time_binned"] = df.event_time.clip(-6, 6)

print(df.event_time_binned.value_counts().sort_index())

In [ ]:
# event_time = -1 is the omitted reference category (Treatment(reference=-1) is the
# Python equivalent of Stata's ib(-1).event_time)
es = smf.ols(
    f"eq_index_mh_all ~ C(event_time_binned, Treatment(reference=-1)) + {controls}"
    " + C(uni_id) + C(survey_wave)", df).fit(
    cov_type="cluster", cov_kwds={"groups": df.uni_id})

# pull out the event-time coefficients and confidence intervals
rows = []
for name in es.params.index:
    if "event_time_binned" in name:
        t = int(name.split("[T.")[1].rstrip("]"))
        lo, hi = es.conf_int().loc[name]
        rows.append({"event_time": t, "coef": es.params[name], "lo": lo, "hi": hi})
ev = pd.DataFrame(rows).sort_values("event_time")
ev.loc[len(ev)] = {"event_time": -1, "coef": 0.0, "lo": 0.0, "hi": 0.0}   # reference period
ev = ev.sort_values("event_time")
ev.round(3)

In [ ]:
plt.figure(figsize=(9, 5))
plt.errorbar(ev.event_time, ev.coef, yerr=[ev.coef - ev.lo, ev.hi - ev.coef],
             fmt="o", capsize=4)
plt.axhline(0, color="black", lw=1)
plt.axvline(-0.5, color="gray", ls="--", lw=1)
plt.xlabel("Event time (survey waves relative to Facebook introduction)")
plt.ylabel("Effect on poor mental health index")
plt.title("Event study: Facebook introduction and student mental health")
plt.show()

**Question 10. Interpret the event-study graph.** What do the pre-treatment coefficients (`event_time` < 0) tell you about the plausibility of parallel trends? What happens from `event_time = 0` onward? Does the effect grow, shrink, or stay flat as exposure lengthens?

**Answer:** *[your answer here]*

## Part 4: Robustness and interpretation

**Question 11.** Re-estimate the Part 2 regression separately for female and male students. **Do the effects differ by gender?** The paper reports effects concentrated among students most susceptible to mental illness — does your (simulated) comparison point in a similar direction?

In [ ]:
by_gender = []
for g, label in [(1, "Female"), (0, "Male")]:
    sub = df[df.female == g]
    r = smf.ols(f"eq_index_mh_all ~ T + {controls} + C(uni_id) + C(survey_wave)", sub).fit(
        cov_type="cluster", cov_kwds={"groups": sub.uni_id})
    by_gender.append(r)
    print(f"{label:7s} effect = {r.params['T']:+.4f}  (se {r.bse['T']:.4f}, n = {int(r.nobs):,})")

**Answer:** *[your answer here]*

**Question 12.** The paper benchmarks its 0.085 SD effect against the effect of job loss on mental health (about 22% as large) and against a roughly 2 percentage point increase in PHQ-9-defined depression off a 25% base rate. **In your own words, why is it useful for a paper to provide this kind of benchmarking, rather than just reporting a standardized effect size on its own?**

**Answer:** *[your answer here]*

**Question 13.** Suppose a critic argued: *"Colleges with more academically demanding programs might have gotten Facebook earlier (through faculty/administrator networks) **and** independently have worsening student mental health over this period for unrelated reasons, like rising academic pressure."* **Which of your fixed effects or design choices does this threaten, and which does it not?**

**Answer:** *[your answer here]*

## Suggested additional questions (optional)

1. **Dose-response.** Re-run using `exposure_months` in place of `T`. Does a dose-response relationship show up in this simulated data?
2. **Placebo test.** Randomly reshuffle `fb_intro_date` across colleges, rebuild the treatment variable, and re-run the DiD. You should find roughly zero. Why is this a useful check on your code and specification, even though it says nothing about the real world?
3. **Staggered DiD econometrics.** Recent work (Goodman-Bacon; Callaway and Sant'Anna; others) shows that two-way fixed effects can behave badly when treatment timing is staggered *and* effects are heterogeneous — the "negative weighting" problem. Explain the concern in a few sentences in your own words. You do not need to implement an alternative estimator.
4. **Mechanisms.** The paper argues the effect works through unfavourable social comparisons, with larger effects for students living off-campus and of lower socioeconomic status. If you had those variables, how would you test this? Sketch the specification.
5. **External validity.** This is US college students in 2004–2008 using an early, simple Facebook. Why might the effect be larger, smaller, or different in kind for (a) teenagers on Instagram/TikTok today, or (b) working adults on LinkedIn?

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS07"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")